In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
metadata = pd.read_csv(
    "../data/archelect_search.csv",
    sep=",",
    on_bad_lines='skip',
    quoting=3,  
    encoding='utf-8'
)

print(f"Nombre de lignes : {len(metadata)}")
print(f"Nombre de colonnes : {len(metadata.columns)}")
print(f"\nColonnes : {metadata.columns.tolist()}")
metadata.head()

In [ ]:
print("Types des colonnes :")
print(metadata.dtypes)
print("\nValeurs manquantes :")
print(metadata.isnull().sum())
print("\nAperçu statistique :")
metadata.describe(include='all')

Maintenant nous allons extraire les différents .zip, "1981legislatives, "1988legislatives", et "1993legislatives". 

In [ ]:
import zipfile

data_dir = Path("../data")
text_dir = data_dir / "text_files"
text_dir.mkdir(exist_ok=True)

zip_files = list(data_dir.glob("*.zip"))
print(f"ZIP trouvés : {[z.name for z in zip_files]}")

# Extraire chaque ZIP
for zf in zip_files:
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(text_dir)
        print(f"{zf.name} : {len(z.namelist())} fichiers extraits")

Maintenant que l'on a décompresser les 3 .zip, nous allons tout regrouper dans un unique Dataframe

In [ ]:
records = []
for txt_path in text_dir.rglob("*.txt"):
    text = txt_path.read_text(encoding='utf-8', errors='replace')
    parts = txt_path.relative_to(text_dir).parts
    records.append({
        'filename': txt_path.stem,
        'year': parts[0] if len(parts) >= 2 else 'unknown',
        'election_type': parts[1] if len(parts) >= 3 else 'unknown',
        'text': text,
        'text_length': len(text)
    })

df_texts = pd.DataFrame(records)
print(f"Nombre de documents texte : {len(df_texts)}")
print(f"\nDocuments par année :")
print(df_texts['year'].value_counts())
print(f"\nLongueur moyenne des textes par année :")
print(df_texts.groupby('year')['text_length'].mean().round(0))
df_texts.head()